# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [2]:
!pip install requests beautifulsoup4 pandas

In [6]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

def fetch_papers(query, total_papers=10000):
    params = {
        "query": query,
        "fields": "paperId,title,year,abstract,authors,venue,citationCount",
        "sort": "citationCount"
    }

    all_papers = []
    token = None

    while len(all_papers) < total_papers:
        if token:
            params["token"] = token

        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print("Error:", response.status_code)
            print(response.text)
            break

        data = response.json()
        papers = data.get("data", [])

        for paper in papers:
            all_papers.append(paper)
            if len(all_papers) >= total_papers:
                break

        token = data.get("token")
        if not token:
            break

        print(f"Collected {len(all_papers)} papers...")
        time.sleep(1)

    return all_papers


# Run the function
query = "machine learning"
papers = fetch_papers(query, total_papers=10000)

Collected 1000 papers...
Collected 2000 papers...
Collected 3000 papers...
Collected 4000 papers...
Collected 5000 papers...
Collected 6000 papers...
Collected 7000 papers...
Collected 8000 papers...
Collected 9000 papers...
Collected 10000 papers...


In [7]:
df = pd.DataFrame(papers)

# Clean authors column
df["authors"] = df["authors"].apply(
    lambda x: "; ".join([a["name"] for a in x]) if isinstance(x, list) else ""
)

df.to_csv("ml_10000_papers.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


In [8]:
from google.colab import files
files.download("ml_10000_papers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [9]:
# Write code for each of the sub parts with proper comments.
!pip -q install nltk

import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [10]:
import pandas as pd

# CHANGE THIS if your filename is different
FILE = "ml_10000_papers.csv"
df = pd.read_csv(FILE)

print("Shape:", df.shape)
print(df.head(2))

# Auto-detect likely text column
candidates = ["abstract", "review", "text", "content", "comment"]
text_col = None
for c in candidates:
    if c in df.columns:
        text_col = c
        break

print("Detected text column:", text_col)

# If not detected, set manually like:
# text_col = "abstract"

Shape: (10000, 8)
                                    paperId  \
0  0000817f14d29ab7febdd976b9a971c81b7de4f6   
1  000099a8f4c7604b553963915427b46ad4b6e491   

                                               title  \
0  Transcriptomics-Guided In Silico Drug Repurpos...   
1  AI in Multidisciplinary Engineering: A Holisti...   

                                               venue    year  citationCount  \
0                                          ACS Omega  2023.0              0   
1  International Research Journal on Advanced Sci...  2025.0              0   

                                       openAccessPdf  \
0  {'url': 'https://doi.org/10.1021/acsomega.3c05...   
1  {'url': '', 'status': None, 'license': None, '...   

                                             authors  \
0  Joyce V. B. Borba; Beatriz Rosa de Azevedo; La...   
1               Dr. G. Gayatri Tanuja; Chethana T. V   

                                            abstract  
0  In tropical and subtropical areas, ma

In [12]:
def show_samples(before_series, after_series, title, n=5):
    out = pd.DataFrame({
        "before": before_series.fillna("").astype(str).head(n),
        "after": after_series.fillna("").astype(str).head(n)
    })
    print("\n" + "="*70)
    print(title)
    print("="*70)
    display(out)

In [13]:
df["step4_lower"] = df[text_col].fillna("").astype(str).str.lower()
show_samples(df[text_col], df["step4_lower"], "(4) Lowercase all texts")


(4) Lowercase all texts


,before,after
0,"In tropical and subtropical areas, malaria sta...","in tropical and subtropical areas, malaria sta..."
1,Artificial Intelligence (AI) has evolved from ...,artificial intelligence (ai) has evolved from ...
2,,
3,"Everywhere across the globe, banks are increas...","everywhere across the globe, banks are increas..."
4,,


In [14]:
import re

def remove_noise(text: str) -> str:
    text = re.sub(r"[^a-z\s]", " ", text)   # keep lowercase letters + spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["step1_no_punct"] = df["step4_lower"].apply(remove_noise)
show_samples(df["step4_lower"], df["step1_no_punct"], "(1) Remove noise (special chars + punctuation)")


(1) Remove noise (special chars + punctuation)


,before,after
0,"in tropical and subtropical areas, malaria sta...",in tropical and subtropical areas malaria stan...
1,artificial intelligence (ai) has evolved from ...,artificial intelligence ai has evolved from a ...
2,,
3,"everywhere across the globe, banks are increas...",everywhere across the globe banks are increasi...
4,,


In [15]:
def remove_numbers(text: str) -> str:
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["step2_no_numbers"] = df["step1_no_punct"].apply(remove_numbers)
show_samples(df["step1_no_punct"], df["step2_no_numbers"], "(2) Remove numbers")


(2) Remove numbers


,before,after
0,in tropical and subtropical areas malaria stan...,in tropical and subtropical areas malaria stan...
1,artificial intelligence ai has evolved from a ...,artificial intelligence ai has evolved from a ...
2,,
3,everywhere across the globe banks are increasi...,everywhere across the globe banks are increasi...
4,,


In [16]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text: str) -> str:
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

df["step3_no_stopwords"] = df["step2_no_numbers"].apply(remove_stopwords)
show_samples(df["step2_no_numbers"], df["step3_no_stopwords"], "(3) Remove stopwords")


(3) Remove stopwords


,before,after
0,in tropical and subtropical areas malaria stan...,tropical subtropical areas malaria stands prof...
1,artificial intelligence ai has evolved from a ...,artificial intelligence ai evolved computation...
2,,
3,everywhere across the globe banks are increasi...,everywhere across globe banks increasingly vie...
4,,


In [17]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stemming(text: str) -> str:
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

df["step5_stem"] = df["step3_no_stopwords"].apply(stemming)
show_samples(df["step3_no_stopwords"], df["step5_stem"], "(5) Stemming")


(5) Stemming


,before,after
0,tropical subtropical areas malaria stands prof...,tropic subtrop area malaria stand profound pub...
1,artificial intelligence ai evolved computation...,artifici intellig ai evolv comput tool transfo...
2,,
3,everywhere across globe banks increasingly vie...,everywher across globe bank increasingli view ...
4,,


In [18]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatization(text: str) -> str:
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

df["step6_lemma"] = df["step3_no_stopwords"].apply(lemmatization)
show_samples(df["step3_no_stopwords"], df["step6_lemma"], "(6) Lemmatization")


(6) Lemmatization


,before,after
0,tropical subtropical areas malaria stands prof...,tropical subtropical area malaria stand profou...
1,artificial intelligence ai evolved computation...,artificial intelligence ai evolved computation...
2,,
3,everywhere across globe banks increasingly vie...,everywhere across globe bank increasingly view...
4,,


In [19]:
df["clean_text"] = df["step6_lemma"]

OUT_FILE = "cleaned_output.csv"
df.to_csv(OUT_FILE, index=False)

print("Saved cleaned file as:", OUT_FILE)
df[["clean_text"]].head(5)

Saved cleaned file as: cleaned_output.csv


,clean_text
0,tropical subtropical area malaria stand profou...
1,artificial intelligence ai evolved computation...
2,
3,everywhere across globe bank increasingly view...
4,


In [20]:
from google.colab import files
files.download("cleaned_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
print(df.shape)
print(df.columns)
df.head(3)

(1, 1)
Index(['clean_text'], dtype='object')


,clean_text
0,eural network alongside advanced deep learning...


In [36]:
df["clean_text"] = df.astype(str).agg(" ".join, axis=1)
df = df[["clean_text"]]   # keep only final column

df["clean_text"].head(5)

/tmp/ipython-input-472/2929818855.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["clean_text"] = df.astype(str).agg(" ".join, axis=1)


,clean_text
0,eural network alongside advanced deep learning...


In [37]:
df.to_csv("cleaned_output_fixed.csv", index=False)
print("Saved cleaned_output_fixed.csv")

Saved cleaned_output_fixed.csv


In [38]:
from google.colab import files
files.download("cleaned_output_fixed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [39]:
TEXT_COL = "clean_text"

In [42]:
import pandas as pd

df = pd.read_csv("cleaned_output_fixed.csv")
print(df.columns)
df.head()

Index(['clean_text'], dtype='object')


,clean_text
0,eural network alongside advanced deep learning...


In [48]:
!pip -q install spacy nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [49]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")

In [50]:
pos_counts = Counter()

texts = df["clean_text"].fillna("").astype(str).tolist()

for doc in nlp.pipe(texts, batch_size=64):
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            pos_counts["Noun"] += 1
        elif token.pos_ in ["VERB", "AUX"]:
            pos_counts["Verb"] += 1
        elif token.pos_ == "ADJ":
            pos_counts["Adjective"] += 1
        elif token.pos_ == "ADV":
            pos_counts["Adverb"] += 1

print("POS Counts:")
print("Nouns:", pos_counts["Noun"])
print("Verbs:", pos_counts["Verb"])
print("Adjectives:", pos_counts["Adjective"])
print("Adverbs:", pos_counts["Adverb"])

POS Counts:
Nouns: 155
Verbs: 39
Adjectives: 58
Adverbs: 8


In [51]:
example_text = df["clean_text"].iloc[0]
doc = nlp(example_text)
sent = list(doc.sents)[0]

print("Sentence:")
print(sent.text)

print("\nDependency Parse:")
for token in sent:
    print(f"{token.text:15} -> {token.head.text:15} | {token.dep_}")

Sentence:
eural network alongside advanced deep learning model convolutional neural network cnn vgg pre trained deep learning architecture model evaluated using precision recall f score accuracy accuracy primary metric comparison experimental result demonstrate traditional machine learning model like random forest logistic regression perform adequately deep learning model particularly cnn vgg excel predictive accuracy performance metric among model cnn vgg deliver superior result vgg slightly outperforming cnn term precision recall due ability leverage pre trained feature deeper architecture finding highlight efficacy deep learning technique especially vgg heart disease prediction emphasizing ability capture complex pattern improve diagnostic accuracy study provides valuable insight potential leveraging state art deep learning architecture enhancing predictive model healthcare application setting stage future real time diagnostic tool heart disease prediction critical area healthcare a

In [53]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [55]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [56]:
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [58]:
!pip -q install requests beautifulsoup4 pandas tqdm

In [59]:
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

BASE = "https://github.com"
START_URL = "https://github.com/marketplace?type=actions"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; class-project-scraper/1.0; +https://github.com/)",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch_page(page_num: int, session: requests.Session) -> str:
    params = {"type": "actions", "page": page_num}
    r = session.get("https://github.com/marketplace", params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def parse_actions(html: str, page_num: int):
    """
    Extract: name, description, url, page
    Strategy:
      - Find action links: <a href="/marketplace/actions/...">
      - For each link, find nearby text for description by looking at its container
    """
    soup = BeautifulSoup(html, "html.parser")

    # All links to action detail pages
    links = soup.select('a[href^="/marketplace/actions/"]')

    items = []
    seen = set()

    for a in links:
        href = a.get("href", "").strip()
        name = a.get_text(" ", strip=True)

        # Filter out empty / duplicates
        if not href or not name:
            continue

        # Unique by href (some duplicates can appear in header/footer)
        if href in seen:
            continue
        seen.add(href)

        # Try to locate a description near the link
        desc = ""
        container = a.find_parent()
        # Walk up a few levels to find a reasonable block, then pick first <p>
        for _ in range(6):
            if container is None:
                break
            p = container.find("p")
            if p and p.get_text(strip=True):
                desc = p.get_text(" ", strip=True)
                break
            container = container.find_parent()

        full_url = urljoin(BASE, href)

        items.append({
            "product_name": name,
            "description": desc,
            "url": full_url,
            "page": page_num
        })

    return items

def scrape_marketplace_actions(target_n=1000, max_pages=500, min_delay=1.0, max_delay=2.0):
    results = []
    session = requests.Session()

    for page in tqdm(range(1, max_pages + 1), desc="Scraping pages"):
        try:
            html = fetch_page(page, session)
            items = parse_actions(html, page)

            # Add items until we reach target
            for it in items:
                results.append(it)
                if len(results) >= target_n:
                    break

            if len(results) >= target_n:
                break

            # If a page returns nothing useful, stop early
            if len(items) == 0:
                print(f"Stopped early: no items found on page {page}")
                break

            time.sleep(random.uniform(min_delay, max_delay))

        except requests.HTTPError as e:
            print(f"HTTP error on page {page}: {e}")
            time.sleep(5)
            continue
        except Exception as e:
            print(f"Error on page {page}: {e}")
            time.sleep(3)
            continue

    df_actions = pd.DataFrame(results).drop_duplicates(subset=["url"])
    return df_actions

df_actions = scrape_marketplace_actions(target_n=1000)

print("Collected rows:", len(df_actions))
df_actions.head(10)

Scraping pages:   8%|▊         | 42/500 [01:33<17:21,  2.27s/it]

HTTP error on page 43: 429 Client Error: Too Many Requests for url: https://github.com/marketplace?type=actions&page=43


Scraping pages:   9%|▊         | 43/500 [01:38<23:38,  3.10s/it]

HTTP error on page 44: 429 Client Error: Too Many Requests for url: https://github.com/marketplace?type=actions&page=44


Scraping pages:  10%|█         | 51/500 [02:00<17:42,  2.37s/it]

Collected rows: 996


,product_name,description,url,page
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,https://github.com/marketplace/actions/truffle...,1
1,Metrics embed,An infographics generator with 40+ plugins and...,https://github.com/marketplace/actions/metrics...,1
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...",https://github.com/marketplace/actions/yq-port...,1
3,Super-Linter,Super-linter is a ready-to-run collection of l...,https://github.com/marketplace/actions/super-l...,1
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes",https://github.com/marketplace/actions/rebuild...,1
5,Gosec Security Checker,Runs the gosec security checker,https://github.com/marketplace/actions/gosec-s...,1
6,Checkout,Checkout a Git repository at a particular version,https://github.com/marketplace/actions/checkout,1
7,OpenCommit — improve commits with AI 🧙,Replaces lame commit messages with meaningful ...,https://github.com/marketplace/actions/opencom...,1
8,SSH Remote Commands,Executing remote ssh commands,https://github.com/marketplace/actions/ssh-rem...,1
9,Claude Code Action Official,General-purpose Claude agent for GitHub PRs an...,https://github.com/marketplace/actions/claude-...,1


In [60]:
df_actions.to_csv("github_marketplace_actions_1000.csv", index=False, encoding="utf-8")
print("Saved: github_marketplace_actions_1000.csv")

Saved: github_marketplace_actions_1000.csv


In [61]:
!pip -q install nltk
import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [62]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text_basic(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()                       # lowercase
    text = re.sub(r"<.*?>", " ", text)             # remove HTML tags (if any)
    text = re.sub(r"[^a-z\s]", " ", text)          # remove special chars & punctuation & numbers
    text = re.sub(r"\s+", " ", text).strip()       # normalize whitespace
    return text

def preprocess(text: str) -> str:
    text = clean_text_basic(text)
    tokens = word_tokenize(text)                   # tokenization
    tokens = [t for t in tokens if t not in stop_words]   # stopwords removal
    tokens = [lemmatizer.lemmatize(t) for t in tokens]    # lemmatization
    return " ".join(tokens)

# Apply on description (you can also apply on product_name if you want)
df_actions["desc_clean"] = df_actions["description"].apply(preprocess)

df_actions[["product_name", "description", "desc_clean"]].head(10)

,product_name,description,desc_clean
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,find verify leaked credential source code
1,Metrics embed,An infographics generator with 40+ plugins and...,infographics generator plugins option display ...
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...",create read update delete merge validate yaml
3,Super-Linter,Super-linter is a ready-to-run collection of l...,super linter ready run collection linters code...
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes",support amlogic rockchip allwinner box
5,Gosec Security Checker,Runs the gosec security checker,run gosec security checker
6,Checkout,Checkout a Git repository at a particular version,checkout git repository particular version
7,OpenCommit — improve commits with AI 🧙,Replaces lame commit messages with meaningful ...,replaces lame commit message meaningful ai gen...
8,SSH Remote Commands,Executing remote ssh commands,executing remote ssh command
9,Claude Code Action Official,General-purpose Claude agent for GitHub PRs an...,general purpose claude agent github pr issue a...


In [63]:
print("Missing values per column:")
print(df_actions.isna().sum())

print("\nDuplicate URLs:", df_actions.duplicated(subset=["url"]).sum())
print("Duplicate product_name:", df_actions.duplicated(subset=["product_name"]).sum())

# Remove duplicates (URL is best unique key)
df_actions = df_actions.drop_duplicates(subset=["url"]).reset_index(drop=True)
print("\nAfter removing duplicates:", df_actions.shape)

Missing values per column:
product_name    0
description     0
url             0
page            0
desc_clean      0
dtype: int64

Duplicate URLs: 0
Duplicate product_name: 0

After removing duplicates: (996, 5)


In [64]:
# URL should start with https://github.com/marketplace/actions/
bad_url = ~df_actions["url"].astype(str).str.startswith("https://github.com/marketplace/actions/")
print("Bad URL rows:", bad_url.sum())

# Description length (detect empty/too short)
df_actions["desc_len"] = df_actions["description"].fillna("").astype(str).str.len()
print("Empty descriptions:", (df_actions["desc_len"] == 0).sum())
print("Very short descriptions (<20 chars):", (df_actions["desc_len"] < 20).sum())

# Optional: drop rows with empty name or url
df_actions = df_actions[df_actions["product_name"].notna() & df_actions["url"].notna()].reset_index(drop=True)
print("After basic validity filter:", df_actions.shape)

df_actions.head(5)

Bad URL rows: 0
Empty descriptions: 0
Very short descriptions (<20 chars): 26
After basic validity filter: (996, 6)


,product_name,description,url,page,desc_clean,desc_len
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,https://github.com/marketplace/actions/truffle...,1,find verify leaked credential source code,54
1,Metrics embed,An infographics generator with 40+ plugins and...,https://github.com/marketplace/actions/metrics...,1,infographics generator plugins option display ...,102
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...",https://github.com/marketplace/actions/yq-port...,1,create read update delete merge validate yaml,67
3,Super-Linter,Super-linter is a ready-to-run collection of l...,https://github.com/marketplace/actions/super-l...,1,super linter ready run collection linters code...,106
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes",https://github.com/marketplace/actions/rebuild...,1,support amlogic rockchip allwinner box,45


In [65]:
df_actions.to_csv("github_actions_cleaned_with_quality.csv", index=False, encoding="utf-8")
print("Saved: github_actions_cleaned_with_quality.csv")

Saved: github_actions_cleaned_with_quality.csv


In [66]:
from google.colab import files
files.download("github_marketplace_actions_1000.csv")
files.download("github_actions_cleaned_with_quality.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [67]:
!pip -q install tweepy pandas nltk

In [68]:
import re
import pandas as pd
import nltk

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [69]:
BEARER_TOKEN = "PASTE_YOUR_BEARER_TOKEN_HERE"

In [70]:
import tweepy

client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)
print("Client created ✅")

Client created ✅


In [71]:
def collect_tweets(hashtags, max_tweets=500):
    """
    Collect tweets via Twitter API v2 recent search.
    - hashtags: list like ["#MachineLearning", "#ArtificialIntelligence"]
    - max_tweets: total tweets across all hashtags
    """
    rows = []
    per_query_target = max(1, max_tweets // len(hashtags))

    for tag in hashtags:
        # Query: hashtag + English + remove retweets for cleaner text
        query = f'{tag} lang:en -is:retweet'
        collected = 0

        # Tweepy v2 pagination
        paginator = tweepy.Paginator(
            client.search_recent_tweets,
            query=query,
            tweet_fields=["created_at", "lang"],
            expansions=["author_id"],
            user_fields=["username"],
            max_results=100,   # max per request
        )

        for resp in paginator:
            if resp.data is None:
                break

            users = {u.id: u.username for u in (resp.includes.get("users", []) if resp.includes else [])}

            for t in resp.data:
                rows.append({
                    "tweet_id": t.id,
                    "username": users.get(t.author_id, ""),
                    "text": t.text,
                    "created_at": t.created_at,
                    "query": tag
                })
                collected += 1
                if collected >= per_query_target:
                    break

            if collected >= per_query_target:
                break

        print(f"Collected {collected} tweets for {tag}")

    df = pd.DataFrame(rows)
    return df

hashtags = ["#MachineLearning", "#ArtificialIntelligence"]  # you can add more tags
df_raw = collect_tweets(hashtags, max_tweets=500)

print("Raw shape:", df_raw.shape)
df_raw.head(5)

Unauthorized: 401 Unauthorized
Unauthorized

In [72]:
print("Bearer token length:", len(BEARER_TOKEN))
print("Starts with:", BEARER_TOKEN[:5])
print("Ends with:", BEARER_TOKEN[-5:])

Bearer token length: 28
Starts with: PASTE
Ends with: _HERE


In [73]:
BEARER_TOKEN = "PASTE_REAL_LONG_BEARER_TOKEN_HERE"

In [74]:
print("Bearer token length:", len(BEARER_TOKEN))

Bearer token length: 33


In [75]:
BEARER_TOKEN = "PUT_YOUR_REAL_BEARER_TOKEN_FROM_TWITTER_PORTAL_HERE"
BEARER_TOKEN = BEARER_TOKEN.strip()

print("Bearer token length:", len(BEARER_TOKEN))

Bearer token length: 51


In [76]:
import tweepy

client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)

try:
    resp = client.search_recent_tweets(query="machine learning lang:en -is:retweet", max_results=10)
    print("✅ Success. Tweets returned:", 0 if resp.data is None else len(resp.data))
except Exception as e:
    print("❌ Error:", type(e).__name__, e)

❌ Error: Unauthorized 401 Unauthorized
Unauthorized


In [77]:
BEARER_TOKEN = BEARER_TOKEN.strip()
print("Bearer token length:", len(BEARER_TOKEN))
print("Starts:", BEARER_TOKEN[:4], "Ends:", BEARER_TOKEN[-4:])

Bearer token length: 51
Starts: PUT_ Ends: HERE


In [78]:
import tweepy

client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)
print("Client created ✅")

Client created ✅


In [79]:
try:
    resp = client.search_recent_tweets(
        query="machine learning lang:en -is:retweet",
        max_results=10
    )
    print("✅ OK. Tweets returned:", 0 if resp.data is None else len(resp.data))
except Exception as e:
    print("❌", type(e).__name__, e)

❌ Unauthorized 401 Unauthorized
Unauthorized


In [80]:
API_KEY = "PASTE_API_KEY"
API_SECRET = "PASTE_API_SECRET"
ACCESS_TOKEN = "PASTE_ACCESS_TOKEN"
ACCESS_SECRET = "PASTE_ACCESS_SECRET"

import tweepy

auth = tweepy.OAuth1UserHandler(API_KEY, API_SECRET, ACCESS_TOKEN, ACCESS_SECRET)
api = tweepy.API(auth)

try:
    user = api.verify_credentials()
    print("✅ OAuth1 OK. Logged in as:", user.screen_name)
except Exception as e:
    print("❌ OAuth1 failed:", type(e).__name__, e)

❌ OAuth1 failed: Unauthorized 401 Unauthorized
89 - Invalid or expired token.


In [81]:
API_KEY = "PASTE_REAL_API_KEY"
API_SECRET = "PASTE_REAL_API_SECRET"
ACCESS_TOKEN = "PASTE_REAL_ACCESS_TOKEN"
ACCESS_SECRET = "PASTE_REAL_ACCESS_SECRET"

# remove accidental spaces
API_KEY = API_KEY.strip()
API_SECRET = API_SECRET.strip()
ACCESS_TOKEN = ACCESS_TOKEN.strip()
ACCESS_SECRET = ACCESS_SECRET.strip()

print(len(API_KEY), len(API_SECRET), len(ACCESS_TOKEN), len(ACCESS_SECRET))

18 21 23 24


In [82]:
import tweepy

auth = tweepy.OAuth1UserHandler(API_KEY, API_SECRET, ACCESS_TOKEN, ACCESS_SECRET)
api = tweepy.API(auth, wait_on_rate_limit=True)

try:
    user = api.verify_credentials()
    print("✅ OAuth1 OK. Logged in as:", user.screen_name)
except Exception as e:
    print("❌ OAuth1 failed:", type(e).__name__, e)

❌ OAuth1 failed: Unauthorized 401 Unauthorized
89 - Invalid or expired token.


In [84]:
API_KEY = "PASTE_REAL_API_KEY"
API_SECRET = "PASTE_REAL_API_SECRET"
ACCESS_TOKEN = "PASTE_REAL_ACCESS_TOKEN"
ACCESS_SECRET = "PASTE_REAL_ACCESS_SECRET"

API_KEY = API_KEY.strip()
API_SECRET = API_SECRET.strip()
ACCESS_TOKEN = ACCESS_TOKEN.strip()
ACCESS_SECRET = ACCESS_SECRET.strip()

print(len(API_KEY), len(API_SECRET), len(ACCESS_TOKEN), len(ACCESS_SECRET))

18 21 23 24


In [85]:
import tweepy

auth = tweepy.OAuth1UserHandler(API_KEY, API_SECRET, ACCESS_TOKEN, ACCESS_SECRET)
api = tweepy.API(auth, wait_on_rate_limit=True)

try:
    user = api.verify_credentials()
    print("✅ SUCCESS. Logged in as:", user.screen_name)
except Exception as e:
    print("❌ FAILED:", type(e).__name__, e)

❌ FAILED: Unauthorized 401 Unauthorized
89 - Invalid or expired token.


In [86]:
import pandas as pd
import tweepy

def collect_tweets_v11(hashtags, total_per_tag=200):
    rows = []
    for tag in hashtags:
        query = f"{tag} -filter:retweets lang:en"
        count = 0

        for tweet in tweepy.Cursor(
            api.search_tweets,
            q=query,
            tweet_mode="extended"
        ).items(total_per_tag):
            rows.append({
                "tweet_id": tweet.id_str,
                "username": tweet.user.screen_name,
                "text": tweet.full_text,
                "query": tag,
                "created_at": tweet.created_at
            })
            count += 1

        print(f"Collected {count} tweets for {tag}")

    return pd.DataFrame(rows)

hashtags = ["#MachineLearning", "#ArtificialIntelligence"]
df_raw = collect_tweets_v11(hashtags, total_per_tag=250)  # total 500

print(df_raw.shape)
df_raw.head()

Unauthorized: 401 Unauthorized
89 - Invalid or expired token.

In [87]:
df_raw.to_csv("tweets_raw.csv", index=False, encoding="utf-8")

from google.colab import files
files.download("tweets_raw.csv")

NameError: name 'df_raw' is not defined

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

This assignment was challenging, especially dealing with API authentication errors and ensuring proper data cleaning, but it helped me understand real-world data collection and preprocessing workflows. I enjoyed building the complete pipeline from scraping to cleaning and felt the time was reasonable, though debugging could be time-consuming.